<h1>This is the script to train the RE Model.</h1>

In [ ]:
#RE MODEL
import json
import torch
import spacy
import random
from collections import defaultdict, Counter
from torch.optim import AdamW
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="spacy")

# Configuration
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
MAX_LENGTH = 356
BATCH_SIZE = 12
EPOCHS = 8
LEARNING_RATE = 2e-5
NEGATIVE_RATIO = 0.2
MARKERS = ["[E1]", "[/E1]", "[E2]", "[/E2]"]

# Initialize spaCy for sentence segmentation
#nlp = spacy.load("en_core_sci_sm", disable=["ner", "lemmatizer"])

class RelationExtractionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.tokenizer = tokenizer
        self.samples = []
        self.label2id = {"no_relation": 0}
        self.id2label = {0: "no_relation"}
        
        # First pass: collect all relation types
        all_relations = set()
        for article in data.values():
            for rel in article["relations"]:
                all_relations.add(rel["predicate"])
        
        # Create label mapping
        for idx, rel in enumerate(sorted(all_relations), 1):
            self.label2id[rel] = idx
            self.id2label[idx] = rel
            
        # Second pass: process data
        self._process_data(data)

    def _process_data(self, data):
        
        for article in tqdm(data.values(), desc="Processing articles"):
            title_text = article["metadata"]["title"]
            abstract_text = article["metadata"]["abstract"]
    
            # Separate entities into those for 'title' vs. 'abstract'
            title_entities = []
            abstract_entities = []
            for ent in article["entities"]:
                if ent["location"] == "title":
                    # Offsets in the title remain as-is
                    e = {
                        "start": ent["start_idx"],
                        "end": ent["end_idx"],
                        "text": ent["text_span"],
                        "label": ent["label"]
                    }
                    title_entities.append(e)
                elif ent["location"] == "abstract":
                    # Offsets in the abstract remain as-is
                    a = {
                        "start": ent["start_idx"],
                        "end": ent["end_idx"],
                        "text": ent["text_span"],
                        "label": ent["label"]
                    }
                    abstract_entities.append(a)
    
            # Separate relations for title vs. abstract
            # We process those in which both subject and object are in the same scope
            title_relations = []
            abstract_relations = []
            for rel in article["relations"]:
                # We check the location from the entity data
                subj_loc = rel["subject_location"]
                obj_loc = rel["object_location"]
    
                # If both are in title, keep it
                if subj_loc == "title" and obj_loc == "title":
                    title_relations.append(rel)
                # If both are in abstract, keep it
                elif subj_loc == "abstract" and obj_loc == "abstract":
                    abstract_relations.append(rel)
    
            # Process the title subset
            pos_samples_title = self._process_relations(title_text, title_entities, title_relations)
            self.samples.extend(pos_samples_title)
            neg_samples_title = self._generate_negative_samples(title_text, title_entities, title_relations)
            self.samples.extend(neg_samples_title)
    
            # Process the abstract subset
            pos_samples_abs = self._process_relations(abstract_text, abstract_entities, abstract_relations)
            self.samples.extend(pos_samples_abs)
            neg_samples_abs = self._generate_negative_samples(abstract_text, abstract_entities, abstract_relations)
            self.samples.extend(neg_samples_abs)

    def _process_relations(self, text, entities, relations):
        samples = []
        entity_map = {(e["start"], e["end"]): e for e in entities}
        
        for rel in relations:
            subj = entity_map.get((rel["subject_start_idx"], rel["subject_end_idx"]))
            obj = entity_map.get((rel["object_start_idx"], rel["object_end_idx"]))
            
            if subj and obj:
                sample = self._create_sample(text, subj, obj, rel["predicate"])
                if sample:
                    samples.append(sample)
                    # If a new predicate label wasn't in label2id initially, add it
                    if rel["predicate"] not in self.label2id:
                        self.label2id[rel["predicate"]] = len(self.label2id)
        return samples

    def _create_sample(self, text, subj, obj, relation):
        # Just do direct offset marking, no spacy sentence check
        if obj["end"] > len(text) or subj["end"] > len(text):
            return None
        marked_text = self._mark_entities_in_whole_text(
            text,
            (subj["start"], subj["end"]),
            (obj["start"], obj["end"])
        )
        return {
            "text": marked_text,
            "label": relation,
            "entity_types": f"{subj['label']}-{obj['label']}"
        }

    def _mark_entities_in_whole_text(self, text, subj_pos, obj_pos):
        # Sort them so the leftmost entity is E1, rightmost is E2
        entities = sorted([(subj_pos, "E1"), (obj_pos, "E2")], key=lambda x: x[0][0])
        
        parts = []
        last_idx = 0
        for (start, end), marker in entities:
            parts.append(text[last_idx:start])
            parts.append(f"[{marker}]")
            parts.append(text[start:end])
            parts.append(f"[/{marker}]")
            last_idx = end
        parts.append(text[last_idx:])
        return "".join(parts)

    def _generate_negative_samples(self, text, entities, relations):
        samples = []
        existing_pairs = set()
        
        # Track existing relations
        for rel in relations:
            key = (rel["subject_start_idx"], rel["subject_end_idx"],
                   rel["object_start_idx"], rel["object_end_idx"])
            existing_pairs.add(key)
        
        # Sort entities by start offset for a consistent order
        entities = sorted(entities, key=lambda x: x["start"])
        for i in range(len(entities)):
            for j in range(i+1, len(entities)):
                if random.random() > NEGATIVE_RATIO:
                    continue
                e1, e2 = entities[i], entities[j]
                key = (e1["start"], e1["end"], e2["start"], e2["end"])
                if key not in existing_pairs:
                    sample = self._create_sample(text, e1, e2, "no_relation")
                    if sample:
                        samples.append(sample)
        return samples

    def _create_label_mappings(self):
        self.id2label = {v: k for k, v in self.label2id.items()}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        encoding = self.tokenizer(
            sample["text"],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.label2id.get(sample["label"], 0), dtype=torch.long)
        }
        
# Now define train() at top-level (not inside the class).
def train():
    # Load data
    with open("/kaggle/input/train-tratinum-json/train_platinum.json") as f:
        data = json.load(f)
    
    # Initialize tokenizer with special markers
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.add_tokens(MARKERS, special_tokens=True)
    
    # Create dataset
    dataset = RelationExtractionDataset(data, tokenizer)
    
    # Prepare stratification mask
    label_counts = Counter()
    for sample in dataset.samples:
        label_counts[sample["label"]] += 1

    stratify_mask = []
    for sample in dataset.samples:
        label = sample["label"]
        if label_counts[label] >= 2:
            stratify_mask.append(label)
        else:
            stratify_mask.append("##RARE##")

    assert all(isinstance(x, str) for x in stratify_mask), "Stratification mask must contain only strings"

    train_data, val_data = train_test_split(
        dataset.samples,
        test_size=0.2,
        random_state=42,
        stratify=stratify_mask,
        shuffle=True
    )

    print("\nClass distribution in splits:")
    # print("Train:", Counter([s["label"] for s in train_data]))
    # print("Validation:", Counter([s["label"] for s in val_data]))

    train_ds = torch.utils.data.Subset(dataset, [i for i, _ in enumerate(train_data)])
    val_ds = torch.utils.data.Subset(dataset, [i for i, _ in enumerate(val_data)])
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    
    # Initialize model
    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(dataset.label2id),
        id2label=dataset.id2label,
        label2id=dataset.label2id
    )
    model.resize_token_embeddings(len(tokenizer))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )
    
    # Training loop
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
                "labels": batch["labels"].to(device)
            }
            outputs = model(**inputs)
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        
        # Validation
        model.eval()
        val_preds = []
        val_labels = []
    
        with torch.no_grad():
            for batch in val_loader:
                inputs = {
                    "input_ids": batch["input_ids"].to(device),
                    "attention_mask": batch["attention_mask"].to(device)
                }
                labels = batch["labels"].to(device)
                
                outputs = model(**inputs)
                preds = torch.argmax(outputs.logits, dim=1)
            
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        all_possible_labels = list(range(len(dataset.label2id)))
        target_names = [dataset.id2label[i] for i in all_possible_labels]
        
        print(f"\nEpoch {epoch+1} Results:")
        print(classification_report(
            val_labels,
            val_preds,
            labels=all_possible_labels,
            target_names=target_names,
            zero_division=0
        ))
    
    # Save model and tokenizer
    model.save_pretrained("RE_platinum")
    tokenizer.save_pretrained("RE_PLATINUM")


if __name__ == "__main__":
    train()

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Processing articles: 100%|██████████| 111/111 [00:00<00:00, 1361.74it/s]



Class distribution in splits:


...

Epoch 6: 100%|██████████| 855/855 [12:51<00:00,  1.11it/s]



Epoch 6 Results:
                   precision    recall  f1-score   support

      no_relation       1.00      0.99      0.99      2247
     administered       0.00      0.00      0.00         2
           affect       0.88      0.97      0.92        29
 change abundance       1.00      0.75      0.86         8
    change effect       1.00      0.33      0.50         3
change expression       0.97      0.97      0.97        30
      compared to       0.00      0.00      0.00         0
           impact       0.95      0.95      0.95        21
        influence       0.92      0.97      0.94        60
         interact       0.89      0.80      0.84        10
             is a       1.00      0.50      0.67         2
     is linked to       0.94      0.94      0.94        50
       located in       0.94      1.00      0.97        46
          part of       0.89      0.89      0.89         9
      produced by       0.00      0.00      0.00         0
           strike       0.00      0.0

Epoch 7: 100%|██████████| 855/855 [12:52<00:00,  1.11it/s]



Epoch 7 Results:
                   precision    recall  f1-score   support

      no_relation       1.00      1.00      1.00      2247
     administered       1.00      0.50      0.67         2
           affect       0.91      1.00      0.95        29
 change abundance       1.00      0.88      0.93         8
    change effect       1.00      1.00      1.00         3
change expression       0.97      1.00      0.98        30
      compared to       0.00      0.00      0.00         0
           impact       1.00      1.00      1.00        21
        influence       0.98      0.97      0.97        60
         interact       1.00      0.80      0.89        10
             is a       1.00      0.50      0.67         2
     is linked to       0.96      0.94      0.95        50
       located in       0.96      1.00      0.98        46
          part of       0.89      0.89      0.89         9
      produced by       0.00      0.00      0.00         0
           strike       0.00      0.0

Epoch 8: 100%|██████████| 855/855 [12:52<00:00,  1.11it/s]


<h1>This is the script to predict Binary tag based relation extraction</h1>

In [1]:
#BINARY BASED PREDICTIONS
import json
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm

def predict_relations(test_data_path, model_path, threshold=0.5):
    # Load model and tokenizer
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # Load test data
    with open(test_data_path) as f:
        test_data = json.load(f)

    # Process each article
    predictions = {}
    for article_id, article in tqdm(test_data.items(), desc="Processing articles"):
        binary_relations = set()  # Using set to avoid duplicates
        
        # Process title and abstract separately
        for section in ["title", "abstract"]:
            text = article["metadata"][section]
            entities = [e for e in article["entities"] if e["location"] == section]
            
            # Generate all possible entity pairs
            for i in range(len(entities)):
                for j in range(len(entities)):
                    if i == j:
                        continue
                        
                    e1 = entities[i]
                    e2 = entities[j]
                    
                    # Create marked text
                    marked_text = mark_entities(text, e1, e2)
                    
                    # Tokenize and predict
                    encoding = tokenizer(
                        marked_text,
                        max_length=256,
                        padding="max_length",
                        truncation=True,
                        return_tensors="pt"
                    ).to(device)

                    with torch.no_grad():
                        outputs = model(**encoding)
                        probs = torch.softmax(outputs.logits, dim=-1)
                        confidence, pred = torch.max(probs, dim=-1)

                    if confidence.item() > threshold and model.config.id2label[pred.item()] != "no_relation":
                        # Store as tuple to maintain order and avoid duplicates
                        pair = (e1["label"], e2["label"])
                        binary_relations.add(pair)

        # Convert set to required format
        predictions[article_id] = {
            "binary_tag_based_relations": [
                {"subject_label": pair[0], "object_label": pair[1]}
                for pair in binary_relations
            ]
        }

    return predictions

def mark_entities(text, e1, e2):
    # Sort entities by start index
    entities = sorted([(e1, "E1"), (e2, "E2")], key=lambda x: x[0]["start_idx"])
    
    marked = []
    last_idx = 0
    for ent, marker in entities:
        start = ent["start_idx"]
        end = ent["end_idx"]
        marked.append(text[last_idx:start])
        marked.append(f"[{marker}]")
        marked.append(text[start:end])
        marked.append(f"[/{marker}]")
        last_idx = end
    marked.append(text[last_idx:])
    
    return "".join(marked)

# Usage
if __name__ == "__main__":
    predictions = predict_relations(
        test_data_path="/kaggle/input/silver-for-re/silver_for_re.json",
        model_path="/kaggle/input/re_merged_v1/tensorflow2/default/1",
        threshold=0.7 #0.6 default
    )

    # Save results
    with open("silver_0.7_merged.json", "w") as f:
        json.dump(predictions, f, indent=2)

    print("Predictions saved to predictions_0.7_merged.json")

Processing articles: 100%|██████████| 499/499 [2:34:10<00:00, 18.54s/it]  

Predictions saved to predictions_0.7_merged.json


<h1>This is the script to predict Ternary tag based relation extraction</h1>

In [2]:
#TERNARY TAG BASED PREDICTIONS
import json
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm

def predict_relations(test_data_path, model_path, threshold=0.5):
    # Load model and tokenizer
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # Load test data
    with open(test_data_path) as f:
        test_data = json.load(f)

    # Process each article
    predictions = {}
    for article_id, article in tqdm(test_data.items(), desc="Processing articles"):
        ternary_relations = set()  # Using set to avoid duplicates
        
        # Process title and abstract separately
        for section in ["title", "abstract"]:
            text = article["metadata"][section]
            entities = [e for e in article["entities"] if e["location"] == section]
            
            # Generate all possible entity pairs
            for i in range(len(entities)):
                for j in range(len(entities)):
                    if i == j:
                        continue
                        
                    e1 = entities[i]
                    e2 = entities[j]
                    
                    # Sort entities by start index
                    sorted_entities = sorted([e1, e2], key=lambda x: x["start_idx"])
                    subject = sorted_entities[0]
                    object_ent = sorted_entities[1]
                    
                    # Create marked text
                    marked_text = mark_entities_sorted(text, subject, object_ent)
                    
                    # Tokenize and predict
                    encoding = tokenizer(
                        marked_text,
                        max_length=256,
                        padding="max_length",
                        truncation=True,
                        return_tensors="pt"
                    ).to(device)

                    with torch.no_grad():
                        outputs = model(**encoding)
                        probs = torch.softmax(outputs.logits, dim=-1)
                        confidence, pred = torch.max(probs, dim=-1)

                    predicted_label = model.config.id2label[pred.item()]
                    if confidence.item() > threshold and predicted_label != "no_relation":
                        relation = (subject["label"], predicted_label, object_ent["label"])
                        ternary_relations.add(relation)

        # Convert set to required format
        predictions[article_id] = {
            "ternary_tag_based_relations": [
                {
                    "subject_label": subj,
                    "predicate": pred,
                    "object_label": obj
                }
                for (subj, pred, obj) in ternary_relations
            ]
        }

    return predictions

def mark_entities_sorted(text, subject, object_ent):
    marked = []
    last_idx = 0

    # Process subject (E1)
    start = subject["start_idx"]
    end = subject["end_idx"]
    marked.append(text[last_idx:start])
    marked.append("[E1]")
    marked.append(text[start:end])
    marked.append("[/E1]")
    last_idx = end

    # Process object (E2)
    start = object_ent["start_idx"]
    end = object_ent["end_idx"]
    marked.append(text[last_idx:start])
    marked.append("[E2]")
    marked.append(text[start:end])
    marked.append("[/E2]")
    last_idx = end

    # Append remaining text after E2
    marked.append(text[last_idx:])

    return "".join(marked)

# Usage
if __name__ == "__main__":
    predictions = predict_relations(
        test_data_path="/kaggle/input/bronze-for-re/bronze_for_re.json",
        model_path="/kaggle/input/re_merged_v1/tensorflow2/default/1",
        threshold=0.6 #0.7 for the best results
    )

    # Save results
    with open("predictions_ternary_bronze.json", "w") as f:
        json.dump(predictions, f, indent=2)

    print("Predictions saved to predictions_ternary_dev.json")

Processing articles: 100%|██████████| 749/749 [3:12:56<00:00, 15.46s/it]  

Predictions saved to predictions_ternary_dev.json


<h1>This is the script to predict Ternary Mention based relation extraction</h1>

In [ ]:
# TERNARY BASED TAG PREDICTIONS
import json
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm

def predict_relations(test_data_path, model_path, threshold=0.5):
    # Load model and tokenizer
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # Load test data
    with open(test_data_path) as f:
        test_data = json.load(f)

    # Process each article
    predictions = {}
    for article_id, article in tqdm(test_data.items(), desc="Processing articles"):
        ternary_relations = []
        
        # Process title and abstract separately
        for section in ["title", "abstract"]:
            text = article["metadata"][section]
            entities = [e for e in article["entities"] if e["location"] == section]
            
            # Generate all possible entity pairs
            for i in range(len(entities)):
                for j in range(len(entities)):
                    if i == j:
                        continue
                        
                    e1 = entities[i]
                    e2 = entities[j]
                    
                    # Sort entities by start index
                    sorted_entities = sorted([e1, e2], key=lambda x: x["start_idx"])
                    subject = sorted_entities[0]
                    object_ent = sorted_entities[1]
                    
                    # Create marked text
                    marked_text = mark_entities_sorted(text, subject, object_ent)
                    
                    # Tokenize and predict
                    encoding = tokenizer(
                        marked_text,
                        max_length=256,
                        padding="max_length",
                        truncation=True,
                        return_tensors="pt"
                    ).to(device)

                    with torch.no_grad():
                        outputs = model(**encoding)
                        probs = torch.softmax(outputs.logits, dim=-1)
                        confidence, pred = torch.max(probs, dim=-1)

                    predicted_label = model.config.id2label[pred.item()]
                    if confidence.item() > threshold and predicted_label != "no_relation":
                        relation_entry = {
                            "subject_text_span": subject["text_span"],
                            "subject_label": subject["label"],
                            "predicate": predicted_label,
                            "object_text_span": object_ent["text_span"],
                            "object_label": object_ent["label"]
                        }
                        ternary_relations.append(relation_entry)

        # Convert to required format
        predictions[article_id] = {
            "ternary_mention_based_relations": ternary_relations
        }

    return predictions

def mark_entities_sorted(text, subject, object_ent):
    marked = []
    last_idx = 0

    # Process subject (E1)
    start = subject["start_idx"]
    end = subject["end_idx"]
    marked.append(text[last_idx:start])
    marked.append("[E1]")
    marked.append(text[start:end])
    marked.append("[/E1]")
    last_idx = end

    # Process object (E2)
    start = object_ent["start_idx"]
    end = object_ent["end_idx"]
    marked.append(text[last_idx:start])
    marked.append("[E2]")
    marked.append(text[start:end])
    marked.append("[/E2]")
    last_idx = end

    marked.append(text[last_idx:])
    return "".join(marked)

if __name__ == "__main__":
    predictions = predict_relations(
        test_data_path="/kaggle/input/gold-for-re/gold_for_re.json",
        model_path="/kaggle/input/re_merged_v1/tensorflow2/default/1",
        threshold=0.7 #This is the best threshold, also 0.6 is good
    )

    # Save results
    with open("predictions_ternarymention_gold.json", "w") as f:
        json.dump(predictions, f, indent=2)

    print("Predictions saved to predictions_ternary_dev.json")

<h3>Test the model trained with the train data on the test data. (in this case it is the dev.json). For each threshold from 0.20 to 1.00 I iterate and I compute the Precision, Recall, F1 Score.</h3>

In [ ]:
import json
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm

def predict_relations(test_data_path, model_path, threshold=0.5):
    # Load model and tokenizer
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # Load test data
    with open(test_data_path) as f:
        test_data = json.load(f)

    # Process each article
    predictions = {}
    for article_id, article in tqdm(test_data.items(), desc="Processing articles"):
        binary_relations = set()  # Using set to avoid duplicates
        
        # Process title and abstract separately
        for section in ["title", "abstract"]:
            text = article["metadata"][section]
            entities = [e for e in article["entities"] if e["location"] == section]
            
            # Generate all possible entity pairs
            for i in range(len(entities)):
                for j in range(len(entities)):
                    if i == j:
                        continue
                        
                    e1 = entities[i]
                    e2 = entities[j]
                    
                    # Create marked text
                    marked_text = mark_entities(text, e1, e2)
                    
                    # Tokenize and predict
                    encoding = tokenizer(
                        marked_text,
                        max_length=256,
                        padding="max_length",
                        truncation=True,
                        return_tensors="pt"
                    ).to(device)

                    with torch.no_grad():
                        outputs = model(**encoding)
                        probs = torch.softmax(outputs.logits, dim=-1)
                        confidence, pred = torch.max(probs, dim=-1)

                    if confidence.item() > threshold and model.config.id2label[pred.item()] != "no_relation":
                        # Store as tuple to maintain order and avoid duplicates
                        pair = (e1["label"], e2["label"])
                        binary_relations.add(pair)

        # Convert set to required format
        predictions[article_id] = {
            "binary_tag_based_relations": [
                {"subject_label": pair[0], "object_label": pair[1]}
                for pair in binary_relations
            ]
        }

    return predictions

def mark_entities(text, e1, e2):
    # Sort entities by start index
    entities = sorted([(e1, "E1"), (e2, "E2")], key=lambda x: x[0]["start_idx"])
    
    marked = []
    last_idx = 0
    for ent, marker in entities:
        start = ent["start_idx"]
        end = ent["end_idx"]
        marked.append(text[last_idx:start])
        marked.append(f"[{marker}]")
        marked.append(text[start:end])
        marked.append(f"[/{marker}]")
        last_idx = end
    marked.append(text[last_idx:])
    
    return "".join(marked)

def eval_submission_6_2_binary_tag_RE(predictions_path, ground_truth, LEGAL_ENTITY_LABELS):
    try:
        with open(predictions_path, 'r', encoding='utf-8') as file:
            predictions = json.load(file)
    except OSError:
        raise OSError(f'Error opening predictions file: {predictions_path}')

    ground_truth_binary_tag_RE = {}
    count_annotated_relations_per_label = {}

    for pmid, article in ground_truth.items():
        ground_truth_binary_tag_RE[pmid] = []
        for relation in article['binary_tag_based_relations']:
            subject_label = str(relation["subject_label"])
            object_label = str(relation["object_label"])
            label = (subject_label, object_label)
            ground_truth_binary_tag_RE[pmid].append(label)
            if label not in count_annotated_relations_per_label:
                count_annotated_relations_per_label[label] = 0
            count_annotated_relations_per_label[label] += 1

    count_predicted_relations_per_label = {label: 0 for label in count_annotated_relations_per_label}
    count_true_positives_per_label = {label: 0 for label in count_annotated_relations_per_label}

    for pmid in predictions:
        try:
            relations = predictions[pmid]['binary_tag_based_relations']
        except KeyError:
            raise KeyError(f'{pmid} missing "binary_tag_based_relations"')

        for relation in relations:
            subject_label = str(relation["subject_label"])
            object_label = str(relation["object_label"])
            if subject_label not in LEGAL_ENTITY_LABELS:
                raise NameError(f'Illegal subject label: {subject_label}')
            if object_label not in LEGAL_ENTITY_LABELS:
                raise NameError(f'Illegal object label: {object_label}')
            label = (subject_label, object_label)
            if label in count_predicted_relations_per_label:
                count_predicted_relations_per_label[label] += 1
            if pmid in ground_truth_binary_tag_RE and label in ground_truth_binary_tag_RE[pmid]:
                count_true_positives_per_label[label] += 1

    # Calculate metrics
    total_annotated = sum(count_annotated_relations_per_label.values())
    total_predicted = sum(count_predicted_relations_per_label.values())
    total_tp = sum(count_true_positives_per_label.values())

    micro_precision = total_tp / (total_predicted + 1e-10)
    micro_recall = total_tp / (total_annotated + 1e-10)
    micro_f1 = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10)

    macro_precision, macro_recall, macro_f1, n = 0, 0, 0, 0
    for label in count_annotated_relations_per_label:
        tp = count_true_positives_per_label[label]
        pred = count_predicted_relations_per_label[label]
        anno = count_annotated_relations_per_label[label]

        prec = tp / (pred + 1e-10)
        rec = tp / (anno + 1e-10)
        f1 = 2 * (prec * rec) / (prec + rec + 1e-10)

        macro_precision += prec
        macro_recall += rec
        macro_f1 += f1
        n += 1

    macro_precision /= n
    macro_recall /= n
    macro_f1 /= n

    return macro_precision, macro_recall, macro_f1, micro_precision, micro_recall, micro_f1

if __name__ == "__main__":
    # Load ground truth data
    GROUND_TRUTH_PATH = "/kaggle/input/devfinal/dev.json"  # Update path
    with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)

    LEGAL_ENTITY_LABELS = [
        "anatomical location", "animal", "bacteria", "biomedical technique",
        "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
        "human", "microbiome", "statistical technique"
    ]

    # Thresholds from 0.20 to 1.00 in steps of 0.05
    thresholds = [round(0.2 + 0.05*i, 2) for i in range(17)]
    results = {}

    for thresh in thresholds:
        print(f"\nEvaluating threshold: {thresh:.2f}")
        # Generate predictions
        preds = predict_relations(
            test_data_path="/kaggle/input/dev-for-re/dev_for_re.json",
            model_path="/kaggle/input/re_merged_v1/tensorflow2/default/1",
            threshold=thresh
        )
        
        # Save predictions
        thresh_str = f"{thresh:.2f}".replace('.', '_')
        preds_path = f"predictions_{thresh_str}.json"
        with open(preds_path, 'w') as f:
            json.dump(preds, f, indent=2)
        
        # Evaluate
        try:
            macro_p, macro_r, macro_f, micro_p, micro_r, micro_f = eval_submission_6_2_binary_tag_RE(
                preds_path, ground_truth, LEGAL_ENTITY_LABELS
            )
            results[thresh] = {
                'macro_precision': macro_p,
                'macro_recall': macro_r,
                'macro_f1': macro_f,
                'micro_precision': micro_p,
                'micro_recall': micro_r,
                'micro_f1': micro_f
            }
            print(f"Threshold {thresh:.2f} Results:")
            print(f"Macro P: {macro_p:.4f}, R: {macro_r:.4f}, F1: {macro_f:.4f}")
            print(f"Micro P: {micro_p:.4f}, R: {micro_r:.4f}, F1: {micro_f:.4f}")
        except Exception as e:
            print(f"Error evaluating {thresh}: {str(e)}")
            results[thresh] = None

    # Save evaluation results
    with open("all_threshold_results.json", 'w') as f:
        json.dump(results, f, indent=2)
    print("All evaluations completed. Results saved to all_threshold_results.json")